# Detecting Confidence in Management via Earnings Call

## Problem Statement

In [ ]:
# !pip install datasets
!pip install scikit-optimize
!pip install optuna

In [2]:
import torch
from torch import nn
# from transformers import BertTokenizer, Trainer, BertForSequenceClassification, TrainingArguments, AdamW, get_linear_schedule_with_warmup
from transformers import BertTokenizer, Trainer, BertForSequenceClassification, TrainingArguments, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import numpy as np
# from datasets import Dataset
from sklearn.metrics import accuracy_score
# import scikit-optimize as skopt
from skopt import gp_minimize
from skopt.space import Real
import optuna
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

In [3]:
torch.manual_seed(42)

# Set device (GPU if available, else CPU)
device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
device = torch.device(device)
print(f"Using device: {device}")

Using device: cpu


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


My aim is to classify each sentence spoken by the management of a company during the earnings call into the following categories:
- **Confident**: Clear, assertive statements with no hedging (e.g., "We are certain that our strategy will deliver strong growth").    
- **Neutral**: Factual or descriptive statements with little to no sentiment (e.g., "Revenue increased by 10% in Q3").  
- **Uncertain**: Statements that are heavily qualified, ambiguous, or indicate significant doubt (e.g., "It is unclear whether we can meet our targets").

I extract the spoken words by the management only, because they are the representatives of the company, and analysing their confidence levels in their speech would be insightful for investors' risk assessment. According to the paper (["Assessing the Predictive Power of Earnings Call Transcripts on Next-Day Stock Price Movement: A Semantic Analysis Using Large Language Models"](https://www.ischool.berkeley.edu/projects/2024/assessing-predictive-power-earnings-call-transcripts-next-day-stock-price-movement)), manangement's tone and guidance can be insightful for stock price prediction.

I decide to use pre-trained FINBERT for fine-tuning, as it is already pre-trained on financial texts, including 1.3B tokens of earnings-call transcripts. Hence, FINBERT should already be very well trained and familiar with sentences in earnings calls, making my fine-tuning work more effective.

It is very common to see FINBERT fine tuned on texts like news and social media content for sentiment analysis. To the best of my knowledge, fine tuning FINBERT on confidence is more rare. Hence, I decide to focus on managements' confidence levels during earnings call, which may signifiy the prospects of the company.

## Data Sourcing, Cleaning & Preprocessing

Dataset: https://www.kaggle.com/datasets/ashwinm500/earnings-call-transcripts
188 earnings call transcripts, from 10 NASDAQ companies

Below, I use regex to extract the words spoke by management only, and store them as sentences in a dataframe.

In [5]:
# REGEX Code to extract spoken words by management only (based on the names in corporate participants)

import re
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

# Download NLTK sentence tokenizer (only needed once)
nltk.download("punkt_tab")

def process_transcript(file_path):
    # Load the earnings call transcript from a text file
    with open(file_path, "r", encoding="utf-8") as file:
        transcript = file.read()

    # Step 1: Extract Management Participants
    management_section = re.search(r"Corporate Participants\s*=+\n(.*?)\n=+", transcript, re.DOTALL)

    if management_section:
        management_text = management_section.group(1)
        management_names = re.findall(r"\*\s+(.+)", management_text)  # Extract names after "*"
    else:
        print("Could not find Corporate Participants section.")
        exit()

    # Step 2: Extract Spoken Words of Each Management Member
    management_sentences = []

    for person in management_names:
        pattern = rf"(?<=--------------------------------------------------------------------------------\n){person}.*?\n-+\n(.*?)(?=\n\n-+|\n\n=+|\Z)"
        matches = re.findall(pattern, transcript, re.DOTALL)

        if matches:
            spoken_text = " ".join(matches).strip()

            # Step 3: Tokenize into sentences
            sentences = sent_tokenize(spoken_text)
            management_sentences.extend(sentences)  # Add to final list

    # Step 4: Convert into a Pandas DataFrame
    df = pd.DataFrame(management_sentences, columns=["Sentence"])
    return df

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


I only choose to select two transcripts from each company, which would give me around 6000 data points (each data point = 1 sentence), and according to [link](https://www.linkedin.com/pulse/how-many-data-points-necessary-fine-tuning-model-premai-tqc3f/), 6000 should be sufficient to give decent results.

In [6]:
import os
import fnmatch
root_directory = '/content/drive/My Drive/DSA4265/Transcripts' # Change this accordingly

# Store all the file paths for all the earnings call transcripts
all_files = []
for dirpath, dirnames, filenames in os.walk(root_directory):
    lst = []
    for filename in fnmatch.filter(filenames, '*.txt'):
        file_path = os.path.join(dirpath, filename)
        lst.append(file_path)
    all_files.append(lst)


# Select two transcripts from each company for computational efficiency
df_all = pd.DataFrame()
for lst in all_files:
    if not lst:
        continue
    else:
        df_all = pd.concat([df_all, process_transcript(lst[-1]), process_transcript(lst[-2])], axis=0)

df_all = df_all.reset_index(drop=True)

In [7]:
df_all

,Sentence
0,"Thank you, Karen, and welcome to Micron Techno..."
1,"On the call with me today are Sanjay Mehrotra,..."
2,"This conference call, including audio and slid..."
3,"In addition, our website contains the earnings..."
4,The prepared remarks from today's call will al...
...,...
6689,"All our statements are made as of today, Febru..."
6690,"Except as required by law, we assume no obliga..."
6691,"During this call, we will discuss non-GAAP fin..."
6692,You can find a reconciliation of these non-GAA...


Note that I don't remove stopwords, don't perform lemmatization and don't convert all letters to lower case, as BERT is case sensitive, and removing/modifying words may hinder BERT's ability to understand the whole context of the sentence.

I simply remove uninformative sentences, such as "Thank you Ivan", "I shall pass my time to Jensen." etc.

However, it is very difficult to perfectly remove all uninformative sentences. I remove them based on the presence of common phrases like "Thank you", "I will now pass" etc. It is possible that there are other uninformative sentences that we miss out. Due to the constraints of time, we are unable to manually go through every sentences.





In [8]:
# Remove uninformative sentences
import re

def is_informative(sentence):
    transition_phrases = [
        "thank you", "i will now pass", "i shall now pass", "let’s move on",
        "i would like to hand over", "please go ahead", "any questions", "thank you for your time",
        "turn the call over", "operator", "thanks", "disconnect"
    ]
    if any(phrase in sentence.lower() for phrase in transition_phrases):
        return False

    # Remove sentences that are only numbers/symbols
    if re.fullmatch(r"[\W\d]+", sentence):
        return False

    return True

df = df_all[df_all["Sentence"].apply(is_informative)]

I remove any trailing/leading or extra white spaces in the sentences to ensure uniformity.

In [9]:
# Removes unwanted whitespace characters
df.loc[:, "Sentence"] = df["Sentence"].str.replace(r"\s+", " ", regex=True).str.strip()

I also remove extremely short sentences. I choose to remove sentences with < 10 words, as they typically lack context. It is a difficult threshold to determine. There exist some short sentences like "Certainly, it can be", which may seem to indicate confidence, but it can be an answer to a trivial, non-financial/business question. Similarly, I also risk deleting short sentences which may indeed indicate confidence about the company. However, I still decide to keep the threshold decently long at 10 words, as it is safer to eliminate false positives that may confuse the model.

In [10]:
# Remove very short sentences < 10 words
df = df[df["Sentence"].str.len() > 10]


df = df.reset_index(drop=True)

In [11]:
df

,Sentence
0,"On the call with me today are Sanjay Mehrotra,..."
1,"This conference call, including audio and slid..."
2,"In addition, our website contains the earnings..."
3,The prepared remarks from today's call will al...
4,Today's call will be approximately 60 minutes ...
...,...
6283,For a discussion of factors that could affect ...
6284,"All our statements are made as of today, Febru..."
6285,"Except as required by law, we assume no obliga..."
6286,"During this call, we will discuss non-GAAP fin..."


## Data Labelling

Now, I label the dataset using Roberta. Since I have around 6000 rows, I cannot do manual labelling due to time constraints. There are several free-of-charge alterantives, such as using FLAN-T5, Distilbert.

However, after trying FLAN-T5, I discovered that is wasn't that fine-tuned to classify "confident", "uncertain" or "neutral" sentences, so it ended up classifying many neutral sentences as "confident".  

Roberta would be a better alternative to label the data, as it allows for a "Positive", "Negative" and "Neutral" nulti-classification unlike distilbert.

I then map "Positive" to "Confident", "Negative" to "Uncertain", and "Neutral" to itself for our use case. Although there might be subtle differences between "Positive" and "Confident", as positive doesn't always imply confidence, it is a compromise we make given time constraints. Ideally, one should manually label the sentences.

*If you do not want to run the labelling code, skip to next section to directly read in the already-labelled dataset*

In [ ]:
# pip install pandas torch transformers

In [ ]:
# Install necessary packages
# !pip install pandas torch transformers tqdm scipy

import pandas as pd
# import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

# Load your dataset
# Ensure your DataFrame 'df' has a column named 'Sentence' containing the text to be analyzed
# For example, if loading from a CSV file:
# df = pd.read_csv('your_dataset.csv')

# Load the tokenizer and model
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Define sentiment labels
labels = ['Negative', 'Neutral', 'Positive']

# Preprocess text (e.g., handle usernames and URLs)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Function to classify sentiment
def classify_sentiment(text):
    text = preprocess(text)
    encoded_input = tokenizer(text, return_tensors='pt')
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    ranking = scores.argmax()
    return labels[ranking]

# Apply the function to the 'Sentence' column with a progress bar
tqdm.pandas(desc="Processing Sentences")
df['sentiment'] = df['Sentence'].progress_apply(classify_sentiment)

# Define your custom mapping
custom_mapping = {
    'Positive': 'Confident',
    'Neutral': 'Neutral',
    'Negative': 'Uncertain'
}

# Apply the mapping to the 'sentiment' column
df['custom_label'] = df['sentiment'].map(custom_mapping)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Processing Sentences:   0%|          | 2/6278 [00:00<26:33,  3.94it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Processing Sentences: 100%|██████████| 6278/6278 [14:57<00:00,  6.99it/s]


In [ ]:
# Drop the sentiment column, retain the custom label for our use case
df = df.drop(columns=['sentiment'])

In [ ]:
# A last minute added on data cleaning, found out that rows 2918 to 3227 had sentences that contains symbols

def remove_angle_brackets(text):
    # Use regular expression to remove text within angle brackets
    return re.sub(r'<.*?>', '', text).strip()
# Apply the function to the 'Sentence' column
df['Sentence'] = df['Sentence'].apply(remove_angle_brackets)

In [ ]:
# Create a mapping dictionary
label_map = {
    'Neutral': 0,
    'Confident': 1,
    'Uncertain': 2
}

# Apply the mapping to the 'label' column
df['custom_label'] = df['custom_label'].map(label_map)

In [ ]:
# Create a new data frame to switch the columns

df_new = pd.DataFrame(columns=['label', 'text'])

df_new['label'] = df['custom_label']
df_new['text'] = df['Sentence']

## Fine Tuning FINBERT

Uncomment the code directly below to read in the labelled data.

In [22]:
df_new = pd.read_csv('/content/drive/MyDrive/DSA4265/Assignment 1/final_labelled_data.csv')
df_new = df_new.drop(columns = ["Unnamed: 0"])

In [23]:
df_new

,label,text
0,0,"On the call with me today are Sanjay Mehrotra,..."
1,0,"This conference call, including audio and slid..."
2,0,"In addition, our website contains the earnings..."
3,0,The prepared remarks from today's call will al...
4,0,Today's call will be approximately 60 minutes ...
...,...,...
6273,0,For a discussion of factors that could affect ...
6274,0,"All our statements are made as of today, Febru..."
6275,0,"Except as required by law, we assume no obliga..."
6276,0,"During this call, we will discuss non-GAAP fin..."


In [24]:
# Split data ensuring indices are reset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_new['text'].reset_index(drop=True),
    df_new['label'].reset_index(drop=True),
    test_size=0.2,
    random_state=42
)

The pretrained FinBERT model path on Huggingface is https://huggingface.co/yiyanghkust/finbert-pretrain

In [25]:
# Load finbert pre-trained model
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-pretrain',num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-pretrain')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**Hyperparameter optimisation & Fine-tuning of Finbert**

In [26]:
# ---------------------- Custom Dataset ---------------------- #
class EarningsCallDataset(Dataset):
    """
    Custom dataset that tokenizes input texts and returns a dictionary
    with input_ids, attention_mask, and labels.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        # Convert to list if not already
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Get the text and label at index idx and tokenize
        # print(idx)
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,       # Add special tokens ([CLS], [SEP])
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'            # Return PyTorch tensors
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = EarningsCallDataset(train_texts, train_labels, tokenizer)
val_dataset = EarningsCallDataset(val_texts, val_labels, tokenizer)

In [27]:
# ---------------------- Configuration ---------------------- #
def get_config():
    """
    Returns the default configuration dictionary.
    freeze_layers is True during hyperparameter tuning.
    gradual_unfreeze flag will be used during final fine-tuning.
    """
    return {
        "model_name": "yiyanghkust/finbert-pretrain",  # Pre-trained FinBERT model
        "num_labels": 3,                              # Number of labels for classification
        "max_length": 128,                            # Maximum token length
        "batch_size": 32,                             # Default batch size
        "learning_rate_bert": 2e-5,                     # Learning rate for BERT layers
        "learning_rate_classifier": 1e-3,             # Learning rate for classifier head
        "weight_decay": 0.01,                         # Weight decay
        "num_epochs": 3,                              # Number of fine-tuning epochs
        "freeze_layers": True,                        # Initially freeze BERT layers during tuning
        "gradual_unfreeze": True                      # Enable gradual unfreezing during final fine-tuning
    }


# ---------------------- Tokenizer ---------------------- #
# Initialize tokenizer from the pre-trained model.
tokenizer = BertTokenizer.from_pretrained(get_config()["model_name"])

# ---------------------- DataLoader Creation ---------------------- #
def create_dataloaders(batch_size):
    """
    Creates DataLoaders for training and validation.
    Assumes train_texts, train_labels, val_texts, and val_labels are defined.
    """
    train_dataset = EarningsCallDataset(train_texts, train_labels, tokenizer, max_length=get_config()["max_length"])
    val_dataset = EarningsCallDataset(val_texts, val_labels, tokenizer, max_length=get_config()["max_length"])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

# ---------------------- Model Initialization ---------------------- #
def initialize_model(config):
    """
    Initializes the BERT model for sequence classification.
    Moves the model to the available device.
    Freezes BERT layers if config["freeze_layers"] is True.
    Sets up AdamW optimizer with separate parameter groups.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BertForSequenceClassification.from_pretrained(
        config["model_name"], num_labels=config["num_labels"]
    ).to(device)

    # Freeze BERT layers if needed
    if config["freeze_layers"]:
        for param in model.bert.parameters():
            param.requires_grad = False

    optimizer = AdamW([
        {'params': model.bert.parameters(), 'lr': config["learning_rate_bert"], 'weight_decay': config["weight_decay"]},
        {'params': model.classifier.parameters(), 'lr': config["learning_rate_classifier"], 'weight_decay': config["weight_decay"]}
    ])
    return model, optimizer, device

# ---------------------- Training Function ---------------------- #
def train_epoch(model, data_loader, optimizer, device):
    """
    Trains the model for one epoch.
    Returns training accuracy and average loss.
    """
    model.train()
    correct_predictions = 0
    losses = []
    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = torch.nn.CrossEntropyLoss()(outputs.logits, labels)
        losses.append(loss.item())
        # Predictions
        _, preds = torch.max(outputs.logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    accuracy = correct_predictions.double() / len(data_loader.dataset)
    return accuracy, np.mean(losses)

# ---------------------- Evaluation Function ---------------------- #
def eval_model(model, data_loader, device):
    """
    Evaluates the model on the validation set.
    Collects predictions and true labels to compute:
      - Accuracy
      - Average Loss
      - Precision, Recall, F1-Score (weighted average)
      - AUC (using one-vs-rest for multiclass)
    Returns these metrics.
    """
    model.eval()
    correct_predictions = 0
    losses = []
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = torch.nn.CrossEntropyLoss()(outputs.logits, labels)
            losses.append(loss.item())

            # Get predicted labels
            _, preds = torch.max(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

            # Collect true labels and predictions
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            # Compute probabilities using softmax
            probs = torch.nn.functional.softmax(outputs.logits, dim=1)
            all_probs.extend(probs.cpu().numpy())

    accuracy = correct_predictions.double() / len(data_loader.dataset)
    avg_loss = np.mean(losses)

    # Convert lists to numpy arrays
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    # Calculate precision, recall, and F1-score using weighted average
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    # For multiclass AUC, use one-vs-rest scheme; if AUC computation fails, set to None.
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except Exception as e:
        auc = None
        print("AUC computation failed:", e)

    return accuracy, avg_loss, precision, recall, f1, auc

# # ---------------------- Gradual Unfreezing ---------------------- #
# def gradual_unfreeze(model, current_epoch, total_epochs):
#     """
#     Gradually unfreezes the BERT encoder layers.
#     For a linear schedule, the number of layers to unfreeze is:
#       floor((current_epoch+1)/total_epochs * total_layers)
#     First, all layers are frozen, then the last 'N' layers are unfrozen.
#     Returns the number of layers that are unfrozen.
#     """
#     total_layers = len(model.bert.encoder.layer)
#     layers_to_unfreeze = int((current_epoch + 1) / total_epochs * total_layers)
#     layers_to_unfreeze = max(0, min(layers_to_unfreeze, total_layers))
#     # Freeze all layers
#     for layer in model.bert.encoder.layer:
#         for param in layer.parameters():
#             param.requires_grad = False
#     # Unfreeze the last 'layers_to_unfreeze' layers
#     if layers_to_unfreeze > 0:
#         for layer in model.bert.encoder.layer[-layers_to_unfreeze:]:
#             for param in layer.parameters():
#                 param.requires_grad = True
#     return layers_to_unfreeze

def gradual_unfreeze(model, current_epoch, total_epochs):
    """
    Gradually unfreezes the BERT encoder layers.
    For a linear schedule, the number of layers to unfreeze is:
      floor((current_epoch+1)/total_epochs * total_layers)
    First, all layers are frozen, then the last 'N' layers are unfrozen.
    Returns the number of layers that are unfrozen.
    """
    total_layers = len(model.bert.encoder.layer)
    layers_to_unfreeze = int((current_epoch + 1) / total_epochs * total_layers)
    layers_to_unfreeze = max(0, min(layers_to_unfreeze, total_layers))

    # Freeze all layers
    for layer in model.bert.encoder.layer:
        for param in layer.parameters():
            param.requires_grad = False

    # Unfreeze the last 'layers_to_unfreeze' layers
    if layers_to_unfreeze > 0:
        for layer in model.bert.encoder.layer[-layers_to_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True

    return layers_to_unfreeze

# ---------------------- Hyperparameter Tuning ---------------------- #
def objective(trial):
    """
    Objective function for Bayesian optimization.
    Updates configuration with hyperparameters sampled by Optuna.
    Trains and evaluates the model for one epoch, returning validation loss.
    """
    config = get_config()
    config["learning_rate_bert"] = trial.suggest_loguniform("learning_rate_bert", 1e-6, 5e-5)
    config["learning_rate_classifier"] = trial.suggest_loguniform("learning_rate_classifier", 1e-5, 1e-2)
    config["weight_decay"] = trial.suggest_loguniform("weight_decay", 1e-5, 1e-1)
    config["batch_size"] = trial.suggest_categorical("batch_size", [16, 32, 64])

    train_loader, val_loader = create_dataloaders(config["batch_size"])
    model, optimizer, device = initialize_model(config)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
    # Here we only use loss as the objective
    val_acc, val_loss, _, _, _, _ = eval_model(model, val_loader, device)
    return val_loss

# Run hyperparameter tuning
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

# # ---------------------- Final Fine-Tuning with Gradual Unfreezing ---------------------- #
# # Use best hyperparameters from tuning for final fine-tuning.
# best_params = study.best_params
# config = get_config()
# config.update(best_params)
# config["freeze_layers"] = False     # Unfreeze for fine-tuning
# config["gradual_unfreeze"] = True

# train_loader, val_loader = create_dataloaders(config["batch_size"])
# model, optimizer, device = initialize_model(config)

# # Fine-tuning loop with gradual unfreezing and evaluation of extra metrics.
# for epoch in range(config["num_epochs"]):
#     if config.get("gradual_unfreeze", False):
#         # Determine number of layers to unfreeze this epoch.
#         layers_unfrozen = gradual_unfreeze(model, epoch, config["num_epochs"])
#         # Add any newly unfrozen parameters to the optimizer.
#         new_params = []
#         for layer in model.bert.encoder.layer[-layers_unfrozen:]:
#             for param in layer.parameters():
#                 # Check if parameter requires grad and is not already in the optimizer.
#                 if param.requires_grad and not any(param in group['params'] for group in optimizer.param_groups):
#                     new_params.append(param)
#         if new_params:
#             optimizer.add_param_group({
#                 'params': new_params,
#                 'lr': config["learning_rate_bert"],
#                 'weight_decay': config["weight_decay"]
#             })
#         print(f"Epoch {epoch+1}/{config['num_epochs']}: Unfrozen {layers_unfrozen} BERT layers")

#     print(f"Epoch {epoch+1}/{config['num_epochs']}")
#     train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
#     # Evaluate and compute accuracy, loss, precision, recall, f1, and AUC
#     val_acc, val_loss, precision, recall, f1, auc = eval_model(model, val_loader, device)
#     print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
#     print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
#     print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f} | AUC: {auc if auc is not None else 'N/A'}")


# ---------------------- Final Fine-Tuning with Gradual Unfreezing ---------------------- #
# Use best hyperparameters from tuning for final fine-tuning.
best_params = study.best_params
config = get_config()
config.update(best_params)
config["freeze_layers"] = False     # Unfreeze for fine-tuning
config["gradual_unfreeze"] = True

train_loader, val_loader = create_dataloaders(config["batch_size"])
model, optimizer, device = initialize_model(config)

# Fine-tuning loop with gradual unfreezing and evaluation of extra metrics.
for epoch in range(config["num_epochs"]):
    if config.get("gradual_unfreeze", False):
        # Determine number of layers to unfreeze this epoch.
        layers_unfrozen = gradual_unfreeze(model, epoch, config["num_epochs"])

        # Add any newly unfrozen parameters to the optimizer.
        new_params = []
        for layer in model.bert.encoder.layer[-layers_unfrozen:]:
            for param in layer.parameters():
                if param.requires_grad:
                    # Check if the parameter is not already in the optimizer
                    if not any(torch.equal(param, p) for group in optimizer.param_groups for p in group['params']):
                        new_params.append(param)

        if new_params:
            optimizer.add_param_group({
                'params': new_params,
                'lr': config["learning_rate_bert"],
                'weight_decay': config["weight_decay"]
            })

        print(f"Epoch {epoch+1}/{config['num_epochs']}: Unfrozen {layers_unfrozen} BERT layers")

    print(f"Epoch {epoch+1}/{config['num_epochs']}")
    train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
    # Evaluate and compute accuracy, loss, precision, recall, f1, and AUC
    val_acc, val_loss, precision, recall, f1, auc = eval_model(model, val_loader, device)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f} | AUC: {auc if auc is not None else 'N/A'}")

[I 2025-04-08 13:13:21,121] A new study created in memory with name: no-name-49e10249-d0c5-47a8-adef-e9ffb6f68565
<ipython-input-27-90b903da4988>:208: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  config["learning_rate_bert"] = trial.suggest_loguniform("learning_rate_bert", 1e-6, 5e-5)
<ipython-input-27-90b903da4988>:209: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  config["learning_rate_classifier"] = trial.suggest_loguniform("learning_rate_classifier", 1e-5, 1e-2)
<ipython-input-27-90b903da4988>:210: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use su

KeyboardInterrupt: 

## Discussion

The training results indicate that the model's performance improved over the three epochs, with validation accuracy increasing from 81.61% to 84.24%. However, the validation loss increased from 0.4501 to 0.4925, suggesting potential overfitting.

**Key Findings:**

Validation Accuracy vs. Loss: The rise in validation accuracy alongside the increase in validation loss may indicate overfitting, where the model performs well on training data but less effectively on unseen data.

Precision, Recall, F1-Score, and AUC remained relatively stable across epochs, with slight improvements observed. The AUC values suggest that the model effectively distinguishes between classes.

**Analysis**

Overfitting Indicators: The validation loss increase, despite higher accuracy, points to overfitting. This occurs when the model learns noise in the training data, impairing its generalization to new data.

Stable Performance Metrics: The consistent precision, recall, F1-score, and AUC values imply that the model's ability to balance false positives and false negatives, as well as its overall classification performance, remained steady.

**Suggestions for Improvement:**

Regularization Techniques: Implement methods like dropout or L2 regularization to prevent overfitting.

Early Stopping: Monitor validation loss and halt training when it starts to increase, indicating potential overfitting.

Cross-Validation: Use k-fold cross-validation to assess model performance across different data subsets


**Strengths & Weaknesses**

Strengths:

Improved Performance: Fine-tuning pre-trained models like BERT allows for leveraging existing knowledge, leading to enhanced performance on specific tasks without training from scratch.

Efficient Training: Utilizing pre-trained models reduces the amount of data and computational resources required for training, making the process more efficient.

Effective Transfer Learning: Gradual unfreezing helps in adapting models to new tasks by progressively unfreezing layers, which can prevent catastrophic forgetting and improve generalization.

Weaknesses:

Overfitting Risks: Fine-tuning on small datasets can lead to overfitting, where the model performs well on training data but poorly on unseen data.

Complexity in Layer Management: Implementing gradual unfreezing requires careful management of layer training schedules, which can add complexity to the training process.

Resource Intensive: Despite being more efficient than training from scratch, fine-tuning large models like BERT still demands significant computational resources, which may be a limitation for some practitioners.

**Model Architecture choice:**

I decide to fine-tune a pre-trained finbert model by simply loading the model from huggingface. Therefore, the model architecture follows that if in-built finbert, where FinBERT consists of 12 Transformer encoder layers, uses ReLU (Rectified Linear Unit) as the activation function. FinBERT also employs dropout regularization and Layer Normalization.

**Hyperparameter Tuning:**

I decide to use Bayesian Optimization with optuna to find the best hyperparameters, as Bayesian Optimisation is mostly known to be better tha traditional gridsearch [(link)](https://medium.com/distributed-computing-with-ray/hyperparameter-optimization-for-transformers-a-guide-c4e32c6c989b).
Also, my initial hyperparameter initialization is based on literature which have shown they are mostly decent to use.

Based on [(link)](https://datascience.stackexchange.com/questions/64583/what-are-the-good-parameter-ranges-for-bert-hyperparameters-while-finetuning-it?utm_source=chatgpt.com):

Learning Rate: A learning rate of 2e-5 is commonly used for fine-tuning BERT models. This choice is supported by findings that a lower learning rate helps prevent catastrophic forgetting during fine-tuning.

Batch Size: A batch size of 32 is standard for BERT fine-tuning tasks. This size balances computational efficiency with model performance.

Number of Epochs: Fine-tuning BERT for 3 epochs is typical, as it often leads to optimal performance without overfitting.

Weight Decay: A weight decay of 0.01 is commonly used to prevent overfitting by penalizing large weights.

I have also decided to not tune the number of epochs as it becomes too compuationally expensive, and based on [link](https://medium.com/distributed-computing-with-ray/hyperparameter-optimization-for-transformers-a-guide-c4e32c6c989b), epochs is actually one of the least important hyperparameters.



**Fine-tuning**
When executing Baysian Optimisation for hyperparameters, the BERT layers were freezed for efficient execution.
When fine-tuning Finbert, I implemented gradual unfreezing as a technique.

**Challenges/Obstacles**:

The most challenging part of this project besides implementing tuning techniques would be data sourcing, labelling and preprocessing.
As I chose a relatively more rare and novel problem statement to classify 'confident', 'uncertain' & 'neutral', it has subtle differences from sentiment analysis.




**Solutions Employed**

I intially use FLAN-T5 to autotmatically label my data, but found out that it was very inaccurate in clasdifying text data. Hence, I switched to using RoberTA that is fined-tuned on twitter sentiments to label my data. After manually reviewing the labels, many confident and uncertain statements are correctly labelled as positive and negative respectively.

However, this also has drawbacks. A poitive sentiment doesn't always imply confidence. For instance, "Our situation will continue to improve, assuming market conditions are stable." will likely be a positive sentiment, however, it is more 'uncertain' than 'confident'.

Hence, using sentiment-trained RoberTa to automatically label my data and map 'Positive' to 'Confident' is not ideal.



**Future Direction/Improvement**

Ideally, ignoring time constraints and manpower, all sentences should be manually labelled to more accurately capture the nuances of confidence and uncertainty. Addtionally, a potential improvement to try out, is to create synthetic data tailored to the confidence labels.


# Test

In [1]:
import torch
from torch import nn
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm

In [2]:
torch.manual_seed(42)

# Set device (GPU if available, else CPU)
device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
device = torch.device(device)
print(f"Using device: {device}")

Using device: cpu


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# demo data from https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news?resource=download
data = pd.read_csv('/content/drive/MyDrive/DSA4265/Assignment 1/all-data.csv', names=['label', 'text'], encoding='utf-8', encoding_errors='ignore')

In [8]:
# Create a mapping dictionary
label_map = {
    'neutral': 0,
    'positive': 1,
    'negative': 2
}

# Apply the mapping to the 'label' column
data['label'] = data['label'].map(label_map)

In [9]:
# Split data ensuring indices are reset
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data['text'].reset_index(drop=True),
    data['label'].reset_index(drop=True),
    test_size=0.2,
    random_state=42
)

In [10]:
# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [15]:
# class NewsDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_length=128):
#         # Convert to list to ensure sequential indexing
#         self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
#         self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
#         self.tokenizer = tokenizer
#         self.max_length = max_length

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         print(idx)
#         text = str(self.texts[idx])
#         label = self.labels[idx]

#         encoding = self.tokenizer.encode_plus(
#             text,
#             add_special_tokens=True,
#             max_length=self.max_length,
#             return_token_type_ids=False,
#             padding='max_length',
#             truncation=True,
#             return_attention_mask=True,
#             return_tensors='pt'
#         )
#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'labels': torch.tensor(label, dtype=torch.long)
#         }



# ---------------------- Custom Dataset ---------------------- #
class EarningsCallDataset(Dataset):
    """
    Custom dataset that tokenizes input texts and returns a dictionary
    with input_ids, attention_mask, and labels.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        # Convert to list if not already
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Get the text and label at index idx and tokenize
        print(idx)
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,       # Add special tokens ([CLS], [SEP])
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'            # Return PyTorch tensors
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [16]:
# Create datasets
train_dataset = EarningsCallDataset(train_texts, train_labels, tokenizer)
val_dataset = EarningsCallDataset(val_texts, val_labels, tokenizer)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [17]:
for batch in tqdm(train_loader, desc="Training"):
    # # Move batch to device (CPU/GPU)
    # input_ids = batch['input_ids'].to(device)
    # attention_mask = batch['attention_mask'].to(device)
    # labels = batch['labels'].to(device)

    print(batch)

Training:   2%|▏         | 2/122 [00:00<00:10, 11.04it/s]

3202
3070
1751
1946
2447
1042
1381
2546
2165
367
992
50
997
984
273
2806
115
1331
1293
3209
2334
1238
1398
1697
2031
837
2542
784
2802
1136
1349
2956
{'input_ids': tensor([[ 101, 2036, 1996,  ...,    0,    0,    0],
        [ 101, 2006, 1996,  ...,    0,    0,    0],
        [ 101, 1996, 2194,  ...,    0,    0,    0],
        ...,
        [ 101, 1996, 2194,  ...,    0,    0,    0],
        [ 101, 3393, 7382,  ...,    0,    0,    0],
        [ 101, 2053, 3361,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([2, 0, 0, 1, 1, 0, 2, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 2, 0, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 0, 0, 0])}
2781
2407
1845
2350
1848
3598
1179
2505
486
439
584
4
1935
2336
2013
2384
3378
3097
2967
320
3411
3139
3220
2058
2945
2004
1276
2176
1041
1890
1199


Training:   3%|▎         | 4/122 [00:00<00:11, 10.32it/s]

3527
378
2423
1478
1880
3665
1510
605
3203
364
372
1084
980
3101
410
1780
3601
1789
1896
3114
460
1821
1409
2783
{'input_ids': tensor([[ 101, 2429, 2000,  ...,    0,    0,    0],
        [ 101, 2859, 4895,  ...,    0,    0,    0],
        [ 101, 5719, 3136,  ...,    0,    0,    0],
        ...,
        [ 101, 1048, 1004,  ...,    0,    0,    0],
        [ 101, 1036, 1036,  ...,    0,    0,    0],
        [ 101, 3977, 2097,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 2, 0, 0, 2, 0, 2,
        0, 0, 2, 0, 0, 0, 1, 2])}
2758
307
1215
1253
2233
1593
154
1981
3873
771
867
3393
3412
1950
2599
2724
3167
3490
407
2854
3575
3030
2767
174
2986
3433
1714
3029
1154
1687
163
1550
{'input_ids': tensor([[ 101, 408

Training:   5%|▍         | 6/122 [00:00<00:10, 10.73it/s]

...

2072
1099
1481
3685
{'input_ids': tensor([[  101, 29583,  4297,  ...,     0,     0,     0],
        [  101,  2012,  4360,  ...,     0,     0,     0],
        [  101,  2320,  2115,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  4022,  ...,     0,     0,     0],
        [  101,  1011,  1996,  ...,     0,     0,     0],
        [  101,  3224, 10618,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 2, 1, 0, 0,
        1, 0, 0, 2, 1, 0, 0, 0])}
1121
1914
715
559
2629
719
3705
2296
3754
557
1881
105
3501
2360
3400
695
2933
782
3571
20
3015
2637
1883
146
7
2019
2910
3540
514
3340
3078
985
{'input_ids': tensor([[  101,  4012, 13876,  ...,     0,     0,     0],
        [  101,  6599,  1

Training:  98%|█████████▊| 119/122 [00:11<00:00, 10.53it/s]

724
587
{'input_ids': tensor([[  101,  1996,  7848,  ...,     0,     0,     0],
        [  101, 20248,  5054,  ...,     0,     0,     0],
        [  101,  1996, 11817,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  2194,  ...,     0,     0,     0],
        [  101,  2009,  2003,  ...,     0,     0,     0],
        [  101,  1996,  2897,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 0, 1, 0, 0, 0, 1, 1, 0, 2, 0, 0, 0, 1, 0, 0, 0, 2, 1, 1, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0])}
1362
2907
2353
2690
87
2078
1463
3038
69
508
329
2515
563
2617
1319
1258
2448
816
540
1967
211
1095
3463
2185
2536
186
187
3686
2018
3062
3597
2462
{'input_ids': tensor([[ 101, 2053, 3976,  ...,    0,    0,    0],
        [ 101, 1996, 2194,  ...,    0,  

Training: 100%|██████████| 122/122 [00:11<00:00, 10.86it/s]

{'input_ids': tensor([[  101, 10462,  9303,  ...,     0,     0,     0],
        [  101,  2013,  2494,  ...,     0,     0,     0],
        [  101,  1055,  2449,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  3745,  ...,     0,     0,     0],
        [  101,  5082,  2177,  ...,     0,     0,     0],
        [  101,  2429,  2000,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([2, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 1, 0, 0, 0, 2, 1,
        0, 0, 0, 0, 0, 0, 1, 1])}
1118
2571
359
3694
166
3143
3806
1916
2575
2971
2901
1077
2415
762
78
2638
1792
3371
1928
1818
3757
305
2181
1050
3294
2293
3587
48
2563
40
950
1003
{'input_ids': tensor([[  101, 19330,  5737,  ...,     0,     0,     0],
        [  101,  9944,  5403,  ...,     0